In [1]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def find_notebooks_dir() -> Path:
    cwd = Path.cwd().resolve()

    for p in [cwd, *cwd.parents]:
        if (p / "dataset").exists() and (p / "figures").exists():
            return p

    for p in [cwd, *cwd.parents]:
        candidate = p / "analysis" / "notebooks"
        if candidate.exists():
            return candidate

    return cwd

NOTEBOOKS_DIR = find_notebooks_dir()
DATASET_PATH = NOTEBOOKS_DIR / "dataset" / "yandex_music_data.json"
FIG_DIR = NOTEBOOKS_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("NOTEBOOKS_DIR:", NOTEBOOKS_DIR)
print("DATASET_PATH:", DATASET_PATH)
print("FIG_DIR:", FIG_DIR)

if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATASET_PATH}")

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

tracks = pd.DataFrame(data.get("tracks", []))
likes = pd.DataFrame(data.get("likes", []))

likes["liked_at"] = pd.to_datetime(likes["liked_at"], utc=True, errors="coerce")
tracks["id"] = tracks["id"].astype("int64")
likes["track_id"] = likes["track_id"].astype("int64")

df = likes.merge(tracks, left_on="track_id", right_on="id", how="left")

df["primary_genre"] = df["primary_genre"].fillna(
    df["genres"].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None)
)

def first_artist(artists):
    if isinstance(artists, list) and len(artists) > 0:
        return artists[0].get("name")
    return None

df["main_artist"] = df["artists"].apply(first_artist)

def savefig(name: str):
    out = FIG_DIR / name
    plt.tight_layout()
    plt.savefig(out, dpi=180, bbox_inches="tight")
    plt.close()
    return out

df.head()


NOTEBOOKS_DIR: C:\Users\kesha\OneDrive\Рабочий стол\BigData\yandex-music-preference-analysis\analysis\notebooks
DATASET_PATH: C:\Users\kesha\OneDrive\Рабочий стол\BigData\yandex-music-preference-analysis\analysis\notebooks\dataset\yandex_music_data.json
FIG_DIR: C:\Users\kesha\OneDrive\Рабочий стол\BigData\yandex-music-preference-analysis\analysis\notebooks\figures


,track_id,liked_at,id,title,duration_ms,explicit,primary_genre,release_year,genres,artists,albums,main_artist
0,143449115,2025-10-03 15:02:01+00:00,143449115,MARTINE ROSE,186890,True,rusrap,2025.0,[rusrap],"[{'id': 13992820, 'name': 'madk1d'}, {'id': 82...","[{'id': 38435712, 'title': 'MARTINE ROSE', 'ge...",madk1d
1,330817,2025-10-03 09:52:49+00:00,330817,Let Down,299260,False,indie,1997.0,[indie],"[{'id': 36825, 'name': 'Radiohead'}]","[{'id': 3389007, 'title': 'OK Computer', 'genr...",Radiohead
2,10169820,2025-09-29 21:03:59+00:00,10169820,Alive,204540,False,electronics,2013.0,[electronics],"[{'id': 111191, 'name': 'Empire Of The Sun'}]","[{'id': 1182273, 'title': 'Ice On The Dune', '...",Empire Of The Sun
3,332895,2025-09-29 14:47:22+00:00,332895,We Are The People,267360,False,electronics,2008.0,[electronics],"[{'id': 111191, 'name': 'Empire Of The Sun'}]","[{'id': 51841, 'title': 'Walking On A Dream', ...",Empire Of The Sun
4,332764,2025-09-28 13:22:49+00:00,332764,Walking On A Dream,196340,False,electronics,2008.0,[electronics],"[{'id': 111191, 'name': 'Empire Of The Sun'}]","[{'id': 51841, 'title': 'Walking On A Dream', ...",Empire Of The Sun


In [2]:
summary = {
    "n_likes": int(df.shape[0]),
    "n_unique_tracks": int(df["track_id"].nunique()),
    "n_unique_artists": int(df["main_artist"].dropna().nunique()),
    "n_unique_genres": int(df["primary_genre"].dropna().nunique()),
    "period_start": df["liked_at"].min(),
    "period_end": df["liked_at"].max(),
}

summary_df = pd.DataFrame([{
    "Лайков": summary["n_likes"],
    "Уникальных треков": summary["n_unique_tracks"],
    "Уникальных исполнителей": summary["n_unique_artists"],
    "Уникальных жанров": summary["n_unique_genres"],
    "Период (с)": str(summary["period_start"]),
    "Период (по)": str(summary["period_end"]),
}])

summary_df


,Лайков,Уникальных треков,Уникальных исполнителей,Уникальных жанров,Период (с),Период (по)
0,1026,1026,493,55,2019-07-18 09:56:38+00:00,2025-10-03 15:02:01+00:00


In [3]:
top_genres = df["primary_genre"].value_counts().head(10)
top_artists = df["main_artist"].value_counts().head(10)

display(pd.DataFrame({"Жанр": top_genres.index, "Лайков": top_genres.values}))
display(pd.DataFrame({"Исполнитель": top_artists.index, "Лайков": top_artists.values}))

user_top5_genres = top_genres.head(5).index.tolist()
user_top5_artists = top_artists.head(5).index.tolist()

print("Top-5 genres:", user_top5_genres)
print("Top-5 artists:", user_top5_artists)


,Жанр,Лайков
0,indie,154
1,rusrap,138
2,rock,99
3,pop,98
4,alternative,64
5,allrock,51
6,rap,49
7,rusrock,45
8,electronics,32
9,ruspop,32


,Исполнитель,Лайков
0,The Doors,50
1,Arctic Monkeys,44
2,Платина,22
3,Placebo,18
4,Kai Angel,15
5,Дора,15
6,Kanye West,13
7,Сплин,13
8,Motorama,13
9,Hans Zimmer,11


Top-5 genres: ['indie', 'rusrap', 'rock', 'pop', 'alternative']
Top-5 artists: ['The Doors', 'Arctic Monkeys', 'Платина', 'Placebo', 'Kai Angel']


In [4]:
g = df["primary_genre"].value_counts().head(8).sort_values()

plt.figure(figsize=(10,6))
plt.barh(g.index, g.values)
plt.title("Портрет слушателя: топ жанров (по лайкам)")
plt.xlabel("Количество лайков")
savefig("profile_top_genres.png")


WindowsPath('C:/Users/kesha/OneDrive/Рабочий стол/BigData/yandex-music-preference-analysis/analysis/notebooks/figures/profile_top_genres.png')

In [5]:
a = df["main_artist"].value_counts().head(10).sort_values()

plt.figure(figsize=(10,6))
plt.barh(a.index, a.values)
plt.title("Портрет слушателя: топ исполнителей (по лайкам)")
plt.xlabel("Количество лайков")
savefig("profile_top_artists.png")


WindowsPath('C:/Users/kesha/OneDrive/Рабочий стол/BigData/yandex-music-preference-analysis/analysis/notebooks/figures/profile_top_artists.png')

In [6]:
tmp = df.dropna(subset=["liked_at"]).copy()
tmp["dow"] = tmp["liked_at"].dt.dayofweek  # 0=Mon
dow_counts = tmp["dow"].value_counts().reindex(range(7), fill_value=0)

labels = ["Пн","Вт","Ср","Чт","Пт","Сб","Вс"]

plt.figure(figsize=(9,4))
plt.bar(labels, dow_counts.values)
plt.title("Портрет слушателя: активность по дням недели (лайки)")
plt.ylabel("Лайков")
savefig("profile_activity_by_weekday.png")


WindowsPath('C:/Users/kesha/OneDrive/Рабочий стол/BigData/yandex-music-preference-analysis/analysis/notebooks/figures/profile_activity_by_weekday.png')

In [7]:
tmp["hour"] = tmp["liked_at"].dt.hour
hour_counts = tmp["hour"].value_counts().sort_index()

plt.figure(figsize=(10,4))
plt.plot(hour_counts.index, hour_counts.values)
plt.title("Портрет слушателя: активность по часам (лайки)")
plt.xlabel("Час суток")
plt.ylabel("Лайков")
plt.xticks(range(0,24,2))
savefig("profile_activity_by_hour.png")


WindowsPath('C:/Users/kesha/OneDrive/Рабочий стол/BigData/yandex-music-preference-analysis/analysis/notebooks/figures/profile_activity_by_hour.png')

In [8]:
years = tracks["release_year"].dropna()
years = pd.to_numeric(years, errors="coerce").dropna().astype(int)

plt.figure(figsize=(10,4))
plt.hist(years, bins=25)
plt.title("Портрет слушателя: распределение треков по году релиза")
plt.xlabel("Год релиза")
plt.ylabel("Количество треков")
savefig("profile_release_year_hist.png")


WindowsPath('C:/Users/kesha/OneDrive/Рабочий стол/BigData/yandex-music-preference-analysis/analysis/notebooks/figures/profile_release_year_hist.png')

In [9]:
explicit_counts = tracks["explicit"].fillna(False).value_counts()

plt.figure(figsize=(6,6))
labels = ["clean", "explicit"] if len(explicit_counts) == 2 else explicit_counts.index.astype(str)
plt.pie(explicit_counts.values, labels=labels, autopct="%1.1f%%")
plt.title("Портрет слушателя: доля explicit-треков")
savefig("profile_explicit_share.png")


WindowsPath('C:/Users/kesha/OneDrive/Рабочий стол/BigData/yandex-music-preference-analysis/analysis/notebooks/figures/profile_explicit_share.png')

In [10]:
# Простые интерпретации для текста
top_genre = top_genres.index[0] if len(top_genres) else "—"
top_artist = top_artists.index[0] if len(top_artists) else "—"

# пик активности по часам
peak_hour = int(hour_counts.idxmax()) if len(hour_counts) else None
peak_hour_text = f"{peak_hour}:00" if peak_hour is not None else "—"

# самый активный день недели
peak_dow = int(dow_counts.idxmax()) if len(dow_counts) else None
peak_dow_text = ["Пн","Вт","Ср","Чт","Пт","Сб","Вс"][peak_dow] if peak_dow is not None else "—"

explicit_share = float(explicit_counts.get(True, 0) / explicit_counts.sum()) if explicit_counts.sum() else 0.0

portrait = [
    f"За период с {summary['period_start'].date()} по {summary['period_end'].date()} собрано {summary['n_likes']} лайков.",
    f"В коллекции: {summary['n_unique_tracks']} уникальных треков, {summary['n_unique_artists']} исполнителей и {summary['n_unique_genres']} жанров.",
    f"Главный жанр по лайкам: {top_genre}. Самый часто встречающийся исполнитель: {top_artist}.",
    f"Наиболее активное время по лайкам: около {peak_hour_text}. Самый активный день недели: {peak_dow_text}.",
    f"Доля explicit-контента в треках: {explicit_share*100:.1f}%.",
    f"По годам релиза можно оценить, к каким музыкальным «эпохам» тяготеет профиль (см. гистограмму релизов)."
]

print("\n".join("• " + s for s in portrait))


• За период с 2019-07-18 по 2025-10-03 собрано 1026 лайков.
• В коллекции: 1026 уникальных треков, 493 исполнителей и 55 жанров.
• Главный жанр по лайкам: indie. Самый часто встречающийся исполнитель: The Doors.
• Наиболее активное время по лайкам: около 10:00. Самый активный день недели: Вт.
• Доля explicit-контента в треках: 23.2%.
• По годам релиза можно оценить, к каким музыкальным «эпохам» тяготеет профиль (см. гистограмму релизов).
